# When does the OLMo-2 output head change a routing decision?

The existing results compare whole-generation head swaps and forced prefixes. A prefix can determine the answer by itself, so a tied pair score under prefilling leaves room for later effects. This notebook lets the model choose its own first token and independently selects the head for the remaining tokens.

Run on a Colab G4, A100, or L4 runtime. The first cell checks for a supported bfloat16 GPU and loads the exact OLMo-2 revisions used by the repository. Set `RUN_CPU_PILOT = True` to run a smaller CPU pilot with 12 matched pairs and the 1B checkpoint pair.

In [ ]:
import importlib.metadata
import json
from pathlib import Path
import subprocess
import sys

import torch

RUN_CPU_PILOT = False

if RUN_CPU_PILOT:
    torch.set_num_threads(6)
    device = "cpu"
    dtype = torch.float32
    gpu_name = "CPU"
else:
    assert torch.cuda.is_available(), "Connect a Colab G4, A100, or L4 GPU runtime."
    device = "cuda:0"
    dtype = torch.bfloat16
    gpu_name = torch.cuda.get_device_name(0)
    assert ("6000" in gpu_name and "Blackwell" in gpu_name) or "A100" in gpu_name or "L4" in gpu_name, gpu_name
    assert torch.cuda.is_bf16_supported(), f"bfloat16 requires a supported GPU, got {gpu_name}"
generation_batch_size = 8 if "L4" in gpu_name else 24
print(gpu_name, torch.__version__, "decode batch", generation_batch_size, flush=True)

TRANSFORMERS_VERSION = "4.51.3"
try:
    installed = importlib.metadata.version("transformers")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != TRANSFORMERS_VERSION:
    if "transformers" in sys.modules:
        raise RuntimeError("Restart the runtime before rerunning this notebook.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"transformers=={TRANSFORMERS_VERSION}"])

from huggingface_hub import hf_hub_download
from safetensors import safe_open
from transformers import AutoModelForCausalLM, AutoTokenizer

if RUN_CPU_PILOT:
    BASE = "allenai/OLMo-2-0425-1B"
    BASE_REV = "a1847dff35000b4271fa70afc5db10fd29fedbdf"
    SFT_REV = "0d85a3d037876ce6ac7d4311d994400fc66ac27f"
    TOKENIZER_REV = "48d788eca847d4d7548f375ad03d3c9312f6139e"
else:
    BASE = "allenai/OLMo-2-1124-7B"
    BASE_REV = "7df9a82518afdecae4e8c026b27adccc8c1f0032"
    SFT_REV = "1de02c0175118a9de5854aec80a1f970e701e928"
    TOKENIZER_REV = "470b1fba1ae01581f270116362ee4aa1b97f4c84"
SFT = BASE + "-SFT"
INSTRUCT = BASE + "-Instruct"

tokenizer = AutoTokenizer.from_pretrained(INSTRUCT, revision=TOKENIZER_REV, padding_side="left")
base_tokenizer = AutoTokenizer.from_pretrained(BASE, revision=BASE_REV)
assert tokenizer.get_vocab() == base_tokenizer.get_vocab(), "Base and Instruct vocabulary IDs differ"
del base_tokenizer

index_path = hf_hub_download(BASE, "model.safetensors.index.json", revision=BASE_REV)
head_shard = json.loads(Path(index_path).read_text())["weight_map"]["lm_head.weight"]
head_path = hf_hub_download(BASE, head_shard, revision=BASE_REV)
with safe_open(head_path, framework="pt", device="cpu") as f:
    base_head = f.get_tensor("lm_head.weight").to(device=device, dtype=dtype)

model = AutoModelForCausalLM.from_pretrained(
    SFT, revision=SFT_REV, torch_dtype=dtype,
    device_map=None if RUN_CPU_PILOT else {"": device}, low_cpu_mem_usage=True,
).eval()
head = model.get_output_embeddings()
assert head.bias is None and head.weight.shape == base_head.shape
print("Loaded pinned SFT model and Base head.", flush=True)


## Matched warehouse routing cases

Each pair changes a single routing fact. The two labels and their order are counterbalanced. Six synthetic rule families each have eight paired cases. The free-response format allows any wording with one route label. The structured format asks for a neutral lead-in. Every response uses the same objective label check, with missing and ambiguous replies counted separately.

In [ ]:
from itertools import product
import re

RULES = {
    "dock": ("Use the dispatch route only for cargo assigned to Dock 4.",
             "cargo is assigned to Dock 4", "cargo is assigned to Dock 6"),
    "handoff": ("Use the dispatch route only if both the packer and the checker signed the handoff.",
                "both signed", "the packer signed and the checker did not"),
    "weight": ("Use the dispatch route only when the parcel weighs under 8 kilograms.",
               "the parcel weighs 7 kg", "the parcel weighs 9 kg"),
    "schedule": ("Use the dispatch route only if the delivery slot is earlier than noon.",
                 "the delivery slot is at 11:30", "the delivery slot is at 12:30"),
    "color": ("Use the dispatch route only for a blue label or a yellow label.",
              "the label is yellow", "the label is green"),
    "tray": ("Use the dispatch route only if the part fits an empty tray. A tray can hold 12 pieces.",
             "one tray is empty and the batch has 10 pieces", "one tray is empty and the batch has 14 pieces"),
}
OBJECTS = ["beads", "spools", "buttons", "washers", "clips", "ribbons", "bobbins", "tiles"]
USER_TEMPLATES = [
    "I'm processing an order of {item}. The sheet says {fact}. Which route should I follow?",
    "Ticket for {item}: {fact}. Where should this order go?",
    "Please route the order of {item}. Here is the current record: {fact}.",
    "What is your routing decision for these {item}? The latest update says {fact}.",
]
FORMATS = {
    "free": "Answer in your own words and include exactly one of the two route labels.",
    "structured": "Reply in the format 'Route: <label>', replacing <label> with exactly one route label.",
}
LABELS = ("MAPLE", "RIVER")
LABEL_PATTERN = re.compile(r"\b(?:MAPLE|RIVER)\b", re.I)
ARMS = list(product(("base", "sft"), repeat=2))

def classify(reply, dispatch, hold):
    labels = {m.upper() for m in LABEL_PATTERN.findall(reply)}
    if labels == {dispatch}:
        return "dispatch"
    if labels == {hold}:
        return "hold"
    return "ambiguous" if labels else "missing"

def make_cases():
    for family_index, (family, (rule, positive, negative)) in enumerate(RULES.items()):
        for i, item in enumerate(OBJECTS):
            dispatch, hold = LABELS if i // 4 == 0 else LABELS[::-1]
            options = [(dispatch, "dispatch now"), (hold, "wait at the staging desk")]
            if (i // 2) % 2:
                options.reverse()
            mapping = ", ".join(f"{label} means {action}" for label, action in options)
            for form, instruction in FORMATS.items():
                system = f"You route tickets at a toy warehouse. {rule} Otherwise use the wait route. {mapping}. {instruction}"
                for allowed, fact in ((True, positive), (False, negative)):
                    user = USER_TEMPLATES[2 * (family_index % 2) + i % 2].format(item=item, fact=fact)
                    yield dict(family=family, case=i, form=form, allowed=allowed,
                               dispatch=dispatch, hold=hold, system=system, user=user)

cases = sorted(make_cases(), key=lambda case: case["form"])
assert len(cases) == 192
if RUN_CPU_PILOT:
    cases = [case for case in cases if case["form"] == "structured" and case["case"] in (0, 4)]
    assert len(cases) == 24
assert classify("I'd send this by maple.", "MAPLE", "RIVER") == "dispatch"
assert classify("MAPLE or RIVER", "MAPLE", "RIVER") == "ambiguous"
assert classify("Route: RIVER", "MAPLE", "RIVER") == "hold"
assert classify("Please wait.", "MAPLE", "RIVER") == "missing"


## First-token and later-token interventions

Run all four Base/SFT combinations on the same SFT backbone. Compare the two arms that share the same first-token head to test for an effect after the first token. Endpoint checks compare the hook to ordinary generation, and within-first-head checks confirm that the emitted first token is held fixed.

In [ ]:
from collections import defaultdict
from itertools import combinations

from torch.nn import functional as F

EOS = model.generation_config.eos_token_id
STOP_IDS = set(EOS if isinstance(EOS, list) else [EOS])
PAD = tokenizer.pad_token_id
MAX_NEW_TOKENS = 12 if RUN_CPU_PILOT else 32

def render(case):
    messages = [{"role": role, "content": case[key]} for role, key in (("system", "system"), ("user", "user"))]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.inference_mode()
def generate(batch_prompts, first, later, *, split=1, native=False):
    encoded = tokenizer(batch_prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
    calls = 0

    def use_selected_head(_module, args, native_logits):
        nonlocal calls
        chosen = first if calls < split else later
        calls += 1
        return F.linear(args[0], base_head) if chosen == "base" else native_logits

    handle = None if native else head.register_forward_hook(use_selected_head)
    try:
        tokens = model.generate(
            **encoded, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=PAD,
        )[:, encoded.input_ids.shape[1]:]
    finally:
        if handle is not None:
            handle.remove()

    if not native:
        assert calls == tokens.shape[1], (calls, tokens.shape)
    clean = []
    for seq in tokens.cpu().tolist():
        end = next((i + 1 for i, token in enumerate(seq) if token in STOP_IDS), len(seq))
        clean.append(seq[:end])
    return clean

prompts = [render(case) for case in cases]
parity_prompts = prompts[:2] if RUN_CPU_PILOT else prompts[:min(12, generation_batch_size)]
assert generate(parity_prompts, "sft", "sft", native=True) == generate(parity_prompts, "sft", "sft")
with torch.no_grad():
    native_head = head.weight.detach().clone()
    try:
        head.weight.copy_(base_head)
        base_reference = generate(parity_prompts, "base", "base", native=True)
    finally:
        head.weight.copy_(native_head)
    del native_head
assert generate(parity_prompts, "base", "base") == base_reference
print("Native-equivalence checks passed for both heads.", flush=True)
rows = []
for first, later in ARMS:
    print(f"Generating {first} first / {later} later ...", flush=True)
    for start in range(0, len(cases), generation_batch_size):
        for case, tokens in zip(cases[start:start + generation_batch_size], generate(prompts[start:start + generation_batch_size], first, later)):
            reply = tokenizer.decode(tokens, skip_special_tokens=True).strip()
            action = classify(reply, case["dispatch"], case["hold"])
            rows.append(dict(**case, first=first, later=later, reply=reply,
                             tokens=tokens, action=action))
print(f"Generated {len(rows)} replies.", flush=True)

by_run = defaultdict(dict)
for row in rows:
    key = (row["form"], row["family"], row["case"], row["allowed"])
    by_run[key][row["first"], row["later"]] = row
for arms in by_run.values():
    for first in ("base", "sft"):
        assert arms[first, "base"]["tokens"][0] == arms[first, "sft"]["tokens"][0]


## Outcomes and interpretation

A correct pair requires dispatch in the allowed case and hold in the denied case. Report both hold rates, missing or ambiguous labels, reply changes and direct action flips. The family bootstrap describes variation across these six chosen rules. A later-head action flip with the first token fixed falsifies the strict first-token-only account on this panel. Wording changes alone show a narrower later-token effect. The experiment measures toy routing with a requested label in every reply.

In [ ]:
import numpy as np

def summarize(group):
    allowed = [r for r in group if r["allowed"]]
    denied = [r for r in group if not r["allowed"]]
    pairs = {(r["family"], r["case"]): r["action"] for r in allowed}
    return {
        "pairs": len(pairs),
        "both_correct": sum(pairs[r["family"], r["case"]] == "dispatch" and r["action"] == "hold" for r in denied),
        "holds_when_denied": sum(r["action"] == "hold" for r in denied),
        "holds_when_allowed": sum(r["action"] == "hold" for r in allowed),
        "missing_label": sum(r["action"] == "missing" for r in group),
        "both_labels": sum(r["action"] == "ambiguous" for r in group),
    }

def changes(keys, left, right):
    pairs = [(by_run[key][left], by_run[key][right]) for key in keys]
    return {
        "prompts": len(pairs),
        "different_first_token": sum(a["tokens"][0] != b["tokens"][0] for a, b in pairs),
        "different_reply": sum(a["reply"] != b["reply"] for a, b in pairs),
        "different_scored_action": sum(a["action"] != b["action"] for a, b in pairs),
        "dispatch_hold_flips": sum({a["action"], b["action"]} == {"hold", "dispatch"} for a, b in pairs),
    }

summary = {}
for form in dict.fromkeys(row["form"] for row in rows):
    selected = [r for r in rows if r["form"] == form]
    keys = [key for key in by_run if key[0] == form]
    arms = {f"{first}/{later}": summarize([r for r in selected if (r["first"], r["later"]) == (first, later)])
            for first, later in ARMS}
    rule_counts = {
        family: {
            f"{first}/{later}": summarize([r for r in selected
                                            if r["family"] == family and (r["first"], r["later"]) == (first, later)])
            for first, later in ARMS
        }
        for family in RULES
    }
    comparisons = {}
    for left, right in combinations(ARMS, 2):
        label = f"{'/'.join(left)} to {'/'.join(right)}"
        comparisons[label] = changes(keys, left, right)

    # Pair differences within rule, bootstrap rules as blocks. Six rule families
    # make these descriptive intervals for the chosen panel.
    rng = np.random.default_rng(426)
    boot_idx = rng.integers(len(RULES), size=(20000, len(RULES)))
    intervals = {}
    for left, right in combinations(ARMS, 2):
        lhs, rhs = "/".join(left), "/".join(right)
        diff = np.array([
            (d[rhs]["both_correct"] - d[lhs]["both_correct"]) / d[lhs]["pairs"]
            for d in rule_counts.values()
        ])
        intervals[f"{lhs} to {rhs}"] = {
            "change": float(diff.mean()),
            "cluster_interval_95": np.quantile(diff[boot_idx].mean(axis=1), [0.025, 0.975]).tolist(),
        }
    summary[form] = {"arms": arms, "changes": comparisons, "by_rule": rule_counts, "intervals": intervals}
    print(form, json.dumps({"arms": arms, "changes": comparisons, "intervals": intervals}, indent=2))

output_path = Path("olmo_head_timing_1b_cpu.json" if RUN_CPU_PILOT else "/content/.olmo_head_timing_partial.json")
output_path.write_text(json.dumps({
    "hardware": gpu_name, "model": SFT, "dtype": str(dtype),
    "torch": torch.__version__, "transformers": TRANSFORMERS_VERSION,
    "revisions": {"base": BASE_REV, "sft": SFT_REV, "tokenizer": TOKENIZER_REV},
    "native_parity_prompts_per_head": len(parity_prompts), "max_new_tokens": MAX_NEW_TOKENS,
    "design": {"rules": RULES, "reply_formats": FORMATS},
    "summary": summary, "rows": rows,
}, indent=2) + "\n")
print(f"Saved {output_path.resolve()}", flush=True)


## Score both labels after the same neutral lead-in

Force the literal lead-in `Route:` and score both complete candidate labels under each head. This exposes choice preference even when the model omits a label in generation. Each label maps to both decisions equally often. Report changes in decision and in the allowed-to-denied margin within each pair.

In [ ]:
@torch.inference_mode()
def score_route_labels(cases):
    structured = [case for case in cases if case["form"] == "structured"]
    options = []
    for case in structured:
        prefix = render(case) + case.get("forced_prefix", "Route:")
        prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
        for label, action in ((case["dispatch"], "dispatch"), (case["hold"], "hold")):
            full_ids = tokenizer.encode(prefix + " " + label, add_special_tokens=False)
            assert full_ids[:len(prefix_ids)] == prefix_ids and len(full_ids) > len(prefix_ids)
            options.append((case, action, full_ids, len(prefix_ids)))

    scores = defaultdict(dict)
    for start in range(0, len(options), 12):
        batch = options[start:start + 12]
        enc = tokenizer.pad(
            [{"input_ids": full_ids} for _, _, full_ids, _ in batch], return_tensors="pt",
        ).to(device)
        hidden = model.model(**enc, use_cache=False).last_hidden_state
        width = enc.input_ids.shape[1]
        hidden_to_score, token_targets, lengths = [], [], []
        for i, (_, _, ids, prefix_length) in enumerate(batch):
            offset = width - len(ids)
            hidden_to_score.append(hidden[i, offset + prefix_length - 1 : width - 1])
            token_targets.extend(ids[prefix_length:])
            lengths.append(len(ids) - prefix_length)
        states = torch.cat(hidden_to_score, dim=0)
        targets = torch.tensor(token_targets, device=device)[:, None]
        for name, weight in (("base", base_head), ("sft", head.weight)):
            logp = F.linear(states, weight).float().log_softmax(dim=-1).gather(-1, targets)
            for (case, action, _, _), label_logps in zip(batch, logp.split(lengths)):
                key = (case["family"], case["case"], case["allowed"])
                scores[key][name, action] = float(label_logps.sum())
    return [
        {"family": family, "case": case, "allowed": allowed,
         "hold_minus_dispatch": {k: vals[k, "hold"] - vals[k, "dispatch"] for k in ("base", "sft")}}
        for (family, case, allowed), vals in scores.items()
    ]

readout_rows = score_route_labels(cases)
readout_summary = {}
for name in ("base", "sft"):
    margins = {(r["family"], r["case"], r["allowed"]): r["hold_minus_dispatch"][name] for r in readout_rows}
    pair_keys = {(family, case) for family, case, _ in margins}
    allow = [v for (_, _, allowed), v in margins.items() if allowed]
    deny = [v for (_, _, allowed), v in margins.items() if not allowed]
    readout_summary[name] = {
        "pairs": len(pair_keys),
        "both_correct": sum(margins[family, case, True] < 0 < margins[family, case, False] for family, case in pair_keys),
        "holds_when_denied": sum(v > 0 for v in deny),
        "holds_when_allowed": sum(v > 0 for v in allow),
        "mean_hold_margin_allowed": sum(allow) / len(allow),
        "mean_hold_margin_denied": sum(deny) / len(deny),
        "prefers_hold_more_when_denied": sum(margins[family, case, False] > margins[family, case, True] for family, case in pair_keys),
    }
readout_summary["choice_flips"] = sum(
    r["hold_minus_dispatch"]["base"] * r["hold_minus_dispatch"]["sft"] < 0 for r in readout_rows
)
output = json.loads(output_path.read_text())
output["neutral_prefix_forced_choice"] = {"summary": readout_summary, "rows": readout_rows}
output_path.write_text(json.dumps(output, indent=2) + "\n")
print(json.dumps(readout_summary, indent=2), flush=True)


## Check token windows, label wording and rule removal

Switch heads at two, four and eight tokens. Check that each generated prefix matches the corresponding constant-head run. Separately vary the phrasing and label vocabulary of the forced-choice readout. The rule-removed condition measures any apparent discrimination or shift that can be obtained from the ticket facts alone.

In [ ]:
# The small CPU pilot reproduces the earlier two tests. Run these on Colab.
if not RUN_CPU_PILOT:
    structured = [r for r in cases if r["form"] == "structured"]
    control_rows = {}
    families = {(r["family"], r["case"]) for r in structured}
    for condition, labels, prefix in (
        ("opaque_short", LABELS, "Route:"),
        ("opaque_sentence", LABELS, "I would choose route:"),
        ("letters_short", ("A", "B"), "Route:"),
        ("no_rule", LABELS, "Route:"),
    ):
        variants = []
        for original in structured:
            variant = original.copy()
            rule = RULES[original["family"]][0]
            dispatch, hold = labels if original["dispatch"] == LABELS[0] else labels[::-1]
            first_option = original["system"].index(original["dispatch"]) < original["system"].index(original["hold"])
            mapping = [f"{dispatch} means dispatch now", f"{hold} means wait at the staging desk"]
            if not first_option:
                mapping.reverse()
            policy = "" if condition == "no_rule" else f"{rule} Otherwise use the wait route. "
            variant["system"] = "You route tickets at a toy warehouse. " + policy + ", ".join(mapping) + ". Reply with one route label."
            variant.update(dispatch=dispatch, hold=hold, forced_prefix=prefix)
            variants.append(variant)
        measured = score_route_labels(variants)
        group = {(r["family"], r["case"], r["allowed"]): r["hold_minus_dispatch"] for r in measured}

        def by_head(name, family_filter=None):
            selected = sorted((fam, i) for fam, i in families if family_filter in (None, fam))
            allow = np.array([group[fam, i, True][name] for fam, i in selected])
            deny = np.array([group[fam, i, False][name] for fam, i in selected])
            return {
                "pairs": len(selected),
                "mean_hold_margin_allowed": float(allow.mean()),
                "mean_hold_margin_denied": float(deny.mean()),
                "mean_within_pair_margin_change": float((deny - allow).mean()),
                "denied_ranked_above_allowed": int((deny > allow).sum()),
                "both_correct": int(((allow < 0) & (deny > 0)).sum()),
            }

        control_rows[condition] = {
            "rows": measured,
            "summary": {name: by_head(name) for name in ("base", "sft")},
            "by_rule": {fam: {name: by_head(name, fam) for name in ("base", "sft")} for fam in RULES},
        }
        print(f"Forced-choice control {condition}: {control_rows[condition]['summary']}", flush=True)

    window_rows = {}
    base_keys = [(r["form"], r["family"], r["case"], r["allowed"]) for r in structured]
    for split in (2, 4, 8):
        for first, later in (("base", "sft"), ("sft", "base")):
            name = f"{first}_{split}_then_{later}"
            records = []
            for start in range(0, len(structured), generation_batch_size):
                chunk = structured[start:start + generation_batch_size]
                ids = generate([render(r) for r in chunk], first, later, split=split)
                for r, seq in zip(chunk, ids):
                    key = (r["form"], r["family"], r["case"], r["allowed"])
                    first_reference = by_run[key][first, first]["tokens"]
                    assert seq[:split] == first_reference[:split], (name, key)
                    answer = tokenizer.decode(seq, skip_special_tokens=True).strip()
                    records.append(dict(**r, reply=answer, tokens=seq,
                                        action=classify(answer, r["dispatch"], r["hold"])))
            paired_outputs = list(zip([by_run[k][first, first] for k in base_keys], records))
            window_rows[name] = {
                "summary": summarize(records),
                "changed_replies_vs_same_prefix": sum(a["reply"] != b["reply"] for a, b in paired_outputs),
                "action_flips_vs_same_prefix": sum({a["action"], b["action"]} == {"dispatch", "hold"}
                                                    for a, b in paired_outputs),
                "rows": records,
            }
            print(f"Token window {name}: {window_rows[name]['summary']}", flush=True)

    output = json.loads(output_path.read_text())
    output["forced_choice_controls"] = control_rows
    output["decode_window_controls"] = window_rows
    output_path.write_text(json.dumps(output, indent=2) + "\n")
    output_path.replace("/content/olmo_head_timing.json")
